# EDA (P0 выгрузки лактаций)

Демо-EDA для структуры P0. Замените `data/raw/demo_lactations_p0.csv` на выгрузку клиента и перезапустите.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "data/raw/demo_lactations_p0.csv"
today = pd.Timestamp("2025-12-20")

df = pd.read_csv(DATA_PATH, parse_dates=["calving_date"])
df.head()

In [ ]:
# Обзор датасета
summary = {
    "rows": len(df),
    "unique_animals": df["animal_id"].nunique(),
    "unique_farms": df["farm"].nunique(),
    "calving_date_min": df["calving_date"].min(),
    "calving_date_max": df["calving_date"].max(),
}
summary

In [ ]:
# Проверки качества
key_cols = ["farm","animal_id","parity","calving_date"]

dup_count = df.duplicated(subset=key_cols).sum()
invalid_milk = (df["milk_305_kg"] <= 0).sum()
invalid_parity = (~df["parity"].between(1, 12)).sum()
future_dates = (df["calving_date"] > today).sum()

dup_count, invalid_milk, invalid_parity, future_dates

In [ ]:
# Пропуски
missing = df.isna().mean().sort_values(ascending=False)
missing

In [ ]:
plt.figure(figsize=(8,4))
missing.plot(kind="bar")
plt.ylabel("Доля пропусков")
plt.title("Пропуски по колонкам (raw)")
plt.tight_layout()
plt.show()

In [ ]:
# Базовая очистка для EDA-графиков (не финальная)
df_clean = df.copy()
df_clean = df_clean[df_clean["milk_305_kg"] > 0]
df_clean = df_clean[df_clean["parity"].between(1, 12)]
df_clean = df_clean[df_clean["calving_date"] <= today]
len(df_clean)

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df_clean["milk_305_kg"], bins=50)
plt.xlabel("milk_305_kg")
plt.ylabel("count")
plt.title("Распределение milk_305_kg (после базовой очистки)")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot по parity
parities = sorted(df_clean["parity"].unique())
data = [df_clean.loc[df_clean["parity"]==p, "milk_305_kg"].values for p in parities]

plt.figure(figsize=(8,4))
plt.boxplot(data, labels=parities, showfliers=False)
plt.xlabel("parity")
plt.ylabel("milk_305_kg")
plt.title("milk_305_kg по номеру лактации (без выбросов на графике)")
plt.tight_layout()
plt.show()

In [ ]:
# Тренд по времени (в целом)
tmp = df_clean.copy()
tmp["month"] = tmp["calving_date"].dt.to_period("M").dt.to_timestamp()
overall = tmp.groupby("month")["milk_305_kg"].mean()

plt.figure(figsize=(9,4))
plt.plot(overall.index, overall.values)
plt.xlabel("month")
plt.ylabel("mean milk_305_kg")
plt.title("Средний milk_305_kg по месяцам (в целом)")
plt.tight_layout()
plt.show()

## Выводы
- Для MVP обязателен слой QC/валидаторов (даты, диапазоны, дубли).
- `parity` и `farm/site` критичны как признаки.
- Нужна time-based валидация из-за сезонности/дрейфа.
- Показатели качества молока часто неполные → не делаем их обязательными для P0.